# Task 00 — Analyze the 400k 5-fold CV results (from SciServer)

📦 This is the **source of the GIVEN facts**. We ran a 400k 5-fold cross-validation on SciServer
(32 cores) with three models — **HGB, RF, MLP** — and saved the **out-of-fold predictions for every
galaxy** plus per-fold metrics. Those artifacts are bundled in a tarball **on GCS** (not in git):

`gs://macrocosm-lewagon/results/outlier_cv_results.tar.gz`  →  `outlier_cv_out/`
- `oof_predictions.parquet` — objid, redshift, fold, **pred_HGB, pred_RF, pred_MLP** (400k rows)
- `metrics_per_fold.csv`, `metrics_summary.csv`, `hard_objids.csv`, `run_info.json`

🎯 You do **not** re-run the 400k training (that's the expensive part). You **analyze the
precomputed predictions**: establish the tabular ceiling, reconstruct where the 6,974 hard set comes
from, and look at how the three models fail.

**Metric:** `σ_MAD = 1.4826·median(|Δz − median(Δz)|)`, `Δz = (z_pred − z_true)/(1+z)`; outlier = |Δz| > 0.05.

In [ ]:
# === download the SciServer results tarball from GCS and load it ===
import os, tarfile
import pandas as pd, numpy as np

GCS_TAR = "macrocosm-lewagon/results/outlier_cv_results.tar.gz"   # gs://<this>
# Colab:  from google.colab import auth; auth.authenticate_user()
# Local:  needs ADC (`gcloud auth application-default login`) OR pass token=<sa_key.json> to GCSFileSystem
if not os.path.isdir("outlier_cv_out"):
    import gcsfs
    gcsfs.GCSFileSystem().get(GCS_TAR, "outlier_cv_results.tar.gz")
    tarfile.open("outlier_cv_results.tar.gz").extractall(".")

R = "outlier_cv_out"
oof     = pd.read_parquet(f"{R}/oof_predictions.parquet")   # objid, redshift, fold, pred_HGB/RF/MLP
summary = pd.read_csv(f"{R}/metrics_summary.csv")
HARD    = set(pd.read_csv(f"{R}/hard_objids.csv")["objid"])
MODELS  = ["HGB", "RF", "MLP"]
print("oof:", oof.shape, "| columns:", oof.columns.tolist(), "| hard set:", len(HARD))

def smad(dz): return 1.4826 * np.median(np.abs(dz - np.median(dz)))
def dz_of(model):
    return (oof[f"pred_{model}"].values - oof["redshift"].values) / (1 + oof["redshift"].values)

❓ **Question (per-model ceiling)** ❓

👇 From `oof_predictions.parquet`, compute each model's **σ_MAD, MAE and outlier rate** over the 400k out-of-fold predictions. Compare to `metrics_summary.csv`. What is the tabular ceiling?

In [ ]:
# YOUR CODE HERE
for m in MODELS:
    dz = dz_of(m)
    # compute sigma_MAD / MAE / outlier_rate ...


❓ **Question (reconstruct the hard set)** ❓

👇 For each model build the outlier mask `|dz| > 0.05`. Take the **intersection of all three**. How many galaxies? Confirm it matches the GIVEN `hard_objids.csv` (6,974).

In [ ]:
# YOUR CODE HERE
masks = {m: np.abs(dz_of(m)) > 0.05 for m in MODELS}


❓ **Question (how do they fail)** ❓

👇 Plot the **Δz distribution** of each model, hard subset vs full population (`oof['objid'].isin(HARD)`). Is the hard set's Δz one-sided or two-sided?

In [ ]:
# YOUR CODE HERE
import matplotlib.pyplot as plt
is_hard = oof['objid'].isin(HARD).values


❓ **Question (model agreement)** ❓

👇 Compute the **pairwise overlap** (Jaccard) of the three models' outlier sets. Do the models fail on the same galaxies or different ones? What does that imply?

In [ ]:
# YOUR CODE HERE


## 📝 Write your report

In [ ]:
# === write_report: run after filling in your results -> regenerates report.md ===
def write_report(title, results: dict, conclusion: str, path="report.md"):
    lines = [f"# {title}", "", "_Auto-generated by task.ipynb._", "", "## Results", ""]
    for k, v in results.items():
        lines.append(f"- **{k}**: {v}")
    lines += ["", "## Conclusion", "", conclusion, ""]
    open(path, "w").write("\n".join(lines))
    print("wrote", path)

In [ ]:
write_report("Task 00 — 400k CV results analysis",
             {"sigma_MAD per model": "FILL IN", "outlier_rate per model": "FILL IN",
              "3-model intersection size": "FILL IN", "matches hard_objids.csv?": "FILL IN",
              "hard dz one/two-sided": "FILL IN", "pairwise outlier Jaccard": "FILL IN"},
             "FILL IN: the tabular ceiling, where the hard set comes from, and whether the three models fail on the same galaxies.")

In [ ]:
# === Commit & push (run last; be on branch 2026.6.16). The tarball stays on GCS, NOT in git. ===
# git pull origin 2026.6.16   first to get everyone else's work.
!git add task.ipynb report.md && git commit -m "00-cv-results-analysis task" && git push origin 2026.6.16